# 04 · Operating-point selection 与报告

本项目固定复现当前阶段口径：2024 年 12 月同时用于选择和评价，统一标记为 **same-sample policy-development diagnostic**。B3 缺失行执行 `excluded_from_risk_and_cash_denominators`，B2 只做边界评价，不制造 same-x realized outcome。`shape_strength` 的 `A_q` 使用 train-only anchor-level curve fit；公开面板没有真实 `x=0` 结果，因此 `B_q` 是 direct quantile model 在 `x=0` 特征副本上的预测。

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'outputs' / '03_stage_contract.json').exists():
            return candidate
    raise FileNotFoundError('run 03_capacity_inversion.ipynb first')


ROOT = find_repo_root()
OUTPUT_DIR = ROOT / 'outputs'
REPORT_DIR = OUTPUT_DIR / 'report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ACTIVE = json.loads((OUTPUT_DIR / '00_active_data.json').read_text(encoding='utf-8'))
STAGE_03 = json.loads((OUTPUT_DIR / '03_stage_contract.json').read_text(encoding='utf-8'))
RUN_MODE = STAGE_03['run_mode']
RISK_BUDGET_BPS = float(STAGE_03['risk_budget_bps'])
CASH_WEIGHTED_VIOLATION_TOLERANCE = 0.05
UNWEIGHTED_VIOLATION_TOLERANCE = 0.06
MATERIAL_BUCKET_VIOLATION_TOLERANCE = 0.08
MATERIAL_BUCKET_MIN_N = 100
MIN_B3_BOUNDARY_COVERAGE_RATE = 0.95
positive_capacity_required = True
EVALUATION_SCOPE = 'same-sample policy-development diagnostic'
SELECTION_EVALUATION_OVERLAP = True
INDEPENDENT_OUT_OF_SAMPLE_POLICY_EVALUATION = False


In [ ]:
capacity = pd.read_parquet(ROOT / STAGE_03['capacity_candidates'])
capacity = capacity[capacity['selection_eligible_solver'].fillna(False)].copy()
b3 = pd.read_parquet(ROOT / ACTIVE['files']['b3'])
b1 = pd.read_parquet(ROOT / ACTIVE['files']['b1'])
for frame in [capacity, b3, b1]:
    frame['date'] = pd.to_datetime(frame['date'], errors='coerce').dt.normalize()
    frame['sym'] = frame['sym'].astype(str)
    frame['side'] = frame['side'].astype(str).str.lower().str.strip()
capacity['quote_strategy'] = capacity['quote_strategy'].astype(str).str.lower().str.strip()
b3['quote_strategy'] = b3['quote_strategy'].astype(str).str.lower().str.strip()

boundary_candidates = ['b3_oracle_refined_x_safe', 'b3_refined_x_safe', 'x_safe_oracle_price_only']
boundary_column = next((column for column in boundary_candidates if column in b3.columns), None)
if boundary_column is None:
    raise ValueError('B3 boundary column not found')
key_columns = ['date', 'sym', 'side', 'quote_strategy']
b3_contract = b3[key_columns + [boundary_column, 'b3_zero_capacity_fallback_flag']].drop_duplicates(key_columns)
b3_contract = b3_contract.rename(columns={boundary_column: 'b3_boundary_x_adv'})
evaluated = capacity.merge(b3_contract, on=key_columns, how='left')
evaluated['b3_boundary_coverage_flag'] = evaluated['b3_boundary_x_adv'].notna()
evaluated['b3_boundary_exceedance_flag'] = np.where(
    evaluated['b3_boundary_coverage_flag'],
    evaluated['x_safe_price_risk'] > evaluated['b3_boundary_x_adv'] + 1e-12,
    np.nan,
)
evaluated['selection_evaluation_overlap'] = SELECTION_EVALUATION_OVERLAP
evaluated['independent_out_of_sample_policy_evaluation'] = INDEPENDENT_OUT_OF_SAMPLE_POLICY_EVALUATION
evaluated['evaluation_scope'] = EVALUATION_SCOPE

bins = [-np.inf, 0.001, 0.0025, 0.005, 0.01, 0.02, 0.05, np.inf]
labels = ['<=0.1%', '0.1-0.25%', '0.25-0.5%', '0.5-1%', '1-2%', '2-5%', '>5%']
evaluated['submitted_ratio_bucket'] = pd.cut(evaluated['x_safe_price_risk'], bins=bins, labels=labels).astype(str)


In [ ]:
identity_columns = [
    'model_family', 'candidate_policy_id', 'quantile_level', 'prediction_stage',
    'calibration_variant', 'floor_variant', 'boundary_solver',
]
total_b3_keys = len(b3_contract)
score_rows = []
bucket_rows = []
for identity, group in evaluated.groupby(identity_columns, dropna=False, sort=False):
    identity_data = dict(zip(identity_columns, identity if isinstance(identity, tuple) else [identity]))
    valid = group[group['b3_boundary_coverage_flag']].copy()
    boundary_coverage_rate = len(valid) / total_b3_keys if total_b3_keys else 0.0
    unweighted_rate = float(valid['b3_boundary_exceedance_flag'].astype(float).mean()) if len(valid) else np.nan
    cash = pd.to_numeric(valid['submitted_cash_price_only'], errors='coerce').fillna(0.0).clip(lower=0.0)
    cash_denominator = float(cash.sum())
    final_cash = pd.to_numeric(valid['submitted_cash_final_with_fill'], errors='coerce').fillna(0.0).clip(lower=0.0)
    final_cash_denominator = float(final_cash.sum())
    n_positive_price_capacity = int(pd.to_numeric(valid['x_safe_price_risk'], errors='coerce').fillna(0.0).gt(0.0).sum())
    n_positive_final_capacity = int(pd.to_numeric(valid['x_safe_final'], errors='coerce').fillna(0.0).gt(0.0).sum())
    cash_weighted_rate = float(cash[valid['b3_boundary_exceedance_flag'].eq(True)].sum() / cash_denominator) if cash_denominator > 0 else 0.0
    material_rates = []
    for (side, bucket), subset in valid.groupby(['side', 'submitted_ratio_bucket'], dropna=False):
        violation_rate = float(subset['b3_boundary_exceedance_flag'].astype(float).mean())
        material = len(subset) >= MATERIAL_BUCKET_MIN_N
        if material:
            material_rates.append(violation_rate)
        bucket_rows.append({
            **identity_data, 'side': side, 'ratio_bucket': bucket, 'n': len(subset),
            'boundary_exceedance_rate': violation_rate, 'material_bucket': material,
            'evaluation_scope': EVALUATION_SCOPE,
        })
    max_material_rate = max(material_rates) if material_rates else 0.0
    eligible = (
        boundary_coverage_rate >= MIN_B3_BOUNDARY_COVERAGE_RATE
        and cash_weighted_rate <= CASH_WEIGHTED_VIOLATION_TOLERANCE
        and unweighted_rate <= UNWEIGHTED_VIOLATION_TOLERANCE
        and max_material_rate <= MATERIAL_BUCKET_VIOLATION_TOLERANCE
        and (not positive_capacity_required or (
            n_positive_price_capacity > 0 and n_positive_final_capacity > 0
            and cash_denominator > 0.0 and final_cash_denominator > 0.0
        ))
    )
    score_rows.append({
        **identity_data, 'n_total_b3_keys': total_b3_keys, 'n_boundary_evaluable': len(valid),
        'b3_boundary_coverage_rate': boundary_coverage_rate,
        'cash_weighted_boundary_exceedance_rate': cash_weighted_rate,
        'unweighted_boundary_exceedance_rate': unweighted_rate,
        'max_material_bucket_exceedance_rate': max_material_rate,
        'submitted_cash_price_only': cash_denominator,
        'submitted_cash_final_with_fill': final_cash_denominator,
        'n_positive_price_capacity': n_positive_price_capacity,
        'n_positive_final_capacity': n_positive_final_capacity,
        'positive_capacity_required': positive_capacity_required,
        'selection_eligible': eligible, 'evaluation_scope': EVALUATION_SCOPE,
        'selection_evaluation_overlap': SELECTION_EVALUATION_OVERLAP,
        'independent_out_of_sample_policy_evaluation': INDEPENDENT_OUT_OF_SAMPLE_POLICY_EVALUATION,
        'b3_boundary_missing_row_policy': 'excluded_from_risk_and_cash_denominators',
    })

scorecard = pd.DataFrame(score_rows).sort_values('submitted_cash_price_only', ascending=False).reset_index(drop=True)
bucket_scorecard = pd.DataFrame(bucket_rows)
eligible_scorecard = scorecard[scorecard['selection_eligible']].copy()
if eligible_scorecard.empty:
    selection_status = 'no_eligible_candidate_fail_closed'
    selected_policy = scorecard.head(0).copy()
    selected_rows = evaluated.head(0).copy()
else:
    selection_status = 'selected_max_submitted_cash_under_constraints'
    selected_policy = eligible_scorecard.head(1).copy()
    selected_id = selected_policy.iloc[0]['candidate_policy_id']
    selected_rows = evaluated[evaluated['candidate_policy_id'].eq(selected_id)].copy()
selected_policy['selection_status'] = selection_status
selected_policy['selection_evaluation_overlap'] = SELECTION_EVALUATION_OVERLAP
selected_policy['independent_out_of_sample_policy_evaluation'] = INDEPENDENT_OUT_OF_SAMPLE_POLICY_EVALUATION
selected_policy['evaluation_scope'] = EVALUATION_SCOPE
selected_rows['selection_status'] = selection_status
scorecard.to_csv(OUTPUT_DIR / '04_candidate_scorecard.csv', index=False)
bucket_scorecard.to_csv(OUTPUT_DIR / '04_candidate_bucket_scorecard.csv', index=False)
selected_policy.to_csv(OUTPUT_DIR / '04_selected_policy.csv', index=False)
selected_rows.to_parquet(OUTPUT_DIR / '04_selected_policy_boundary_rows.parquet', index=False, compression='zstd')


In [ ]:
common_keys = b3_contract[['date', 'sym', 'side']].drop_duplicates()
b1_common = b1.merge(common_keys.assign(_common=True), on=['date', 'sym', 'side'], how='inner')
b3_realized_column = 'actual_total_bad_move_bps' if 'actual_total_bad_move_bps' in b3.columns else 'b3_total_bad_move_bps'
baseline_rows = [{
    'baseline': 'B1_zero_info_10bps_protection', 'evaluation_method': 'realized_same_x_outcome',
    'n': len(b1_common),
    'mean_submitted_x_adv': float(pd.to_numeric(b1_common['submitted_x_adv'], errors='coerce').mean()),
    'budget_violation_rate': float(pd.to_numeric(b1_common.get('budget_violation_flag', b1_common.get('budget_violation')), errors='coerce').mean()),
    'evaluation_scope': EVALUATION_SCOPE,
}, {
    'baseline': 'B3_adaptive_refined_oracle', 'evaluation_method': 'realized_same_x_outcome_with_future_information',
    'n': len(b3),
    'mean_submitted_x_adv': float(pd.to_numeric(b3[boundary_column], errors='coerce').mean()),
    'budget_violation_rate': float((pd.to_numeric(b3[b3_realized_column], errors='coerce') > RISK_BUDGET_BPS).mean()),
    'evaluation_scope': EVALUATION_SCOPE,
}]
if len(selected_policy):
    selected = selected_policy.iloc[0]
    baseline_rows.append({
        'baseline': 'B2_selected_model_boundary', 'evaluation_method': 'b3_10bps_capacity_boundary_not_same_x_outcome',
        'n': int(selected['n_boundary_evaluable']),
        'mean_submitted_x_adv': float(selected_rows['x_safe_price_risk'].mean()),
        'budget_violation_rate': float(selected['unweighted_boundary_exceedance_rate']),
        'evaluation_scope': EVALUATION_SCOPE,
    })
baseline_comparison = pd.DataFrame(baseline_rows)
baseline_comparison['selection_evaluation_overlap'] = SELECTION_EVALUATION_OVERLAP
baseline_comparison['independent_out_of_sample_policy_evaluation'] = INDEPENDENT_OUT_OF_SAMPLE_POLICY_EVALUATION
baseline_comparison.to_csv(OUTPUT_DIR / '04_baseline_comparison.csv', index=False)

sns.set_theme(style='whitegrid')
fig, axis = plt.subplots(figsize=(9, 5))
sns.scatterplot(
    data=scorecard, x='unweighted_boundary_exceedance_rate', y='submitted_cash_price_only',
    hue='model_family', style='selection_eligible', ax=axis,
)
axis.axvline(UNWEIGHTED_VIOLATION_TOLERANCE, color='red', linestyle='--', linewidth=1)
axis.set_title('Submitted cash vs B3 boundary exceedance (same-sample diagnostic)')
fig.tight_layout()
fig.savefig(REPORT_DIR / 'candidate_cash_vs_boundary_exceedance.png', dpi=150)
plt.close(fig)

report_summary = {
    'run_mode': RUN_MODE, 'evaluation_scope': EVALUATION_SCOPE,
    'strength_label_granularity': STAGE_03['strength_label_granularity'],
    'base_label_source': STAGE_03['base_label_source'],
    'selection_status': selection_status,
    'selection_evaluation_overlap': SELECTION_EVALUATION_OVERLAP,
    'independent_out_of_sample_policy_evaluation': INDEPENDENT_OUT_OF_SAMPLE_POLICY_EVALUATION,
    'b3_boundary_missing_row_policy': 'excluded_from_risk_and_cash_denominators',
    'risk_budget_bps': RISK_BUDGET_BPS,
    'candidate_count': len(scorecard), 'eligible_candidate_count': len(eligible_scorecard),
    'positive_capacity_required': positive_capacity_required,
    'selected_candidate_policy_id': selected_policy.iloc[0]['candidate_policy_id'] if len(selected_policy) else None,
    'interpretation': 'stage-level offline evidence; not independent policy holdout or production guarantee',
}
(OUTPUT_DIR / '04_report_summary.json').write_text(json.dumps(report_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(scorecard.head(12).to_string(index=False))
print(baseline_comparison.to_string(index=False))
print(json.dumps(report_summary, ensure_ascii=False, indent=2))
